# 第五阶段：多模态前置与图文对齐 (CLIP 微调实战)

在前四节课中，我们分别独立学习了“视觉 (CV)”和“语言 (NLP)”。但在真实世界中，人类是同时通过眼睛和耳朵来认识世界的。

2021 年，OpenAI 提出了震惊 AI 界的 **CLIP (Contrastive Language-Image Pre-training)** 模型，彻底打通了视觉和语言的壁垒。它能将“一张狗的图片”和“一句关于狗的文字”映射到高维空间中的同一个点。从此，**“以文搜图”** 和 **“零样本分类 (Zero-shot)”** 成为现实，也为后来的 LLaVA 等前沿多模态大模型奠定了基石。

本节课，我们将使用 Hugging Face 加载 CLIP 模型，并带你手写**对比学习的核心损失函数 (InfoNCE Loss)**，完成一次图文对齐模型的微调！

## 步骤 1-3：多模态处理器 (Processor) ➔ Dataset ➔ DataLoader

### ① 知识讲解
在单模态任务中，图像有 `transforms`，文本有 `Tokenizer`。而在 CLIP 中，我们需要同时处理两者。Hugging Face 提供了 **`Processor`**，它内部同时包装了 `ImageProcessor` (裁剪、归一化) 和 `Tokenizer` (文本分词)。

### ② 为什么这么做
图文对齐任务的数据集必须是严格的**“图文对 (Image-Caption Pair)”**。模型必须知道“哪张图”对应“哪句话”。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
import os

# ③ 完整代码
# 加载预训练的 CLIP 模型和对应的多模态处理器
model_id = "openai/clip-vit-base-patch32" 
processor = CLIPProcessor.from_pretrained(model_id)

# 模拟一个包含“图片”和对应“描述文字”的图文对数据集
class PairDataset(Dataset):
    def __init__(self, processor, num_samples=8):
        self.processor = processor
        self.num_samples = num_samples
        # 模拟文本库
        self.captions = ["a photo of a cat", "a photo of a dog"] * (num_samples // 2)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 模拟读取一张真实图片 (这里我们用随机噪点图代替)
        img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        image = Image.fromarray(img_array)
        text = self.captions[idx]
        
        # ④ 代码逐行解释: Processor 会同时处理图和文
        inputs = self.processor(text=text, images=image, return_tensors="pt", padding="max_length", truncation=True, max_length=16)
        
        # processor 返回的默认都有 batch 维度 [1, ...]，所以需要去掉
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

train_loader = DataLoader(PairDataset(processor, 32), batch_size=4, shuffle=True)
val_loader = DataLoader(PairDataset(processor, 8), batch_size=4, shuffle=False)
print("多模态图文对数据集已准备就绪！")

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))"), '(Request ID: d4212f97-c5b8-4a51-a2ec-502cde8406cc)')' thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json
Retrying in 1s [Retry 1/5].


多模态图文对数据集已准备就绪！


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：千万保证同一个 Batch 里，第 `i` 张图片必定对应第 `i` 句话！打乱数据（`shuffle=True`）时，图片和文字的对应关系绝对不能错位，否则对比学习将完全崩溃。
- **⑥ 科研实验室**：学术界通常使用 `LAION-400M` 这样的超大数据集（包含 4 亿对从网页上爬取的 `<img>` 和 `alt` 文本）来从零训练 CLIP。
- **⑦ 工程实践建议**：工业界做“以文搜图”电商搜索引擎时，就是把商品图片全部提特征存入向量数据库（如 Milvus），然后把用户的搜索词也提特征，利用余弦相似度极速召回商品。

## 步骤 4：模型结构揭秘 (双塔架构)

### ① 知识讲解
CLIP 是经典的**双塔模型 (Two-Tower Architecture)**：
1. **视觉塔 (Vision Model)**：一个 Vision Transformer (ViT) 或 ResNet，把图片变成一个长度为 512 的一维向量。
2. **文本塔 (Text Model)**：一个 Transformer，把句子变成同样长度为 512 的一维向量。
3. **最后**，计算这两个向量的余弦相似度。越相似越好。

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = CLIPModel.from_pretrained(model_id).to(device)

# 冻结文本塔，只微调视觉塔（在某些垂直领域比如医疗影像，文本语义不需要变，只需让图片适应文本）
# 这种“冻一塔、训一塔”的操作在工程中非常常见
for param in model.text_model.parameters():
    param.requires_grad = False

print("CLIP 双塔模型加载完毕，且已冻结 Text Encoder，准备微调 Vision Encoder。")

CLIP 双塔模型加载完毕，且已冻结 Text Encoder，准备微调 Vision Encoder。


## 步骤 5-6：训练 (Train) 与 对比学习损失函数 (InfoNCE)

### ① 知识讲解与 ② 动机
在之前的单模态分类中，我们的标签是 `0, 1, 2`。但在图文对齐中，我们没有这种绝对标签！
我们只有相对关系：**“这批数据中，图片 A 和文字 A 是一对（正样本），图片 A 和文字 B 是一对错的（负样本）”**。

对比学习（Contrastive Learning）的核心哲学就是：**拉近正样本的距离，推开负样本的距离**。为了让你深刻理解，我不使用 Hugging Face 内置的 Loss，而是带你手搓大名鼎鼎的 `InfoNCE Loss`（这也正是 CLIP 论文原版公式）。

In [5]:
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

# 对比学习的本质实际上是被转换成了多分类交叉熵计算
loss_fn = nn.CrossEntropyLoss()

epochs = 2
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        optimizer.zero_grad()
        
        # 1. 抽取两种模态的特征特征 (已经过了投影层，维度一致)
        # 比如 batch_size=4 时，维度都是 [4, 512]
        image_features = model.get_image_features(pixel_values=pixel_values)
        text_features = model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        
        # 2. 必须进行 L2 归一化，这样点积（dot product）就完全等价于余弦相似度（Cosine Similarity）
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        
        # 3. 计算 相似度矩阵 (Similarity Matrix)
        # logit_scale 是模型内部自带的一个可学习温度系数，用于放大相似度的差异
        logit_scale = model.logit_scale.exp()
        # [4, 512] @ [512, 4] -> [4, 4]
        # 矩阵的对角线元素 (0,0), (1,1) 是对应的正样本图文对；其余全是非匹配的负样本！
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()
        
        # 4. 计算 InfoNCE 对比损失
        # 我们的目标是让对角线的值（正确匹配）尽可能大。
        # 巧妙之处：我们把这就当成一个有 4 个类别的分类问题！真实的 label 就是 0, 1, 2, 3（即对角线的索引）
        labels = torch.arange(len(logits_per_image), dtype=torch.long, device=device)
        
        loss_i = loss_fn(logits_per_image, labels)
        loss_t = loss_fn(logits_per_text, labels)
        loss = (loss_i + loss_t) / 2 # 对称损失
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Contrastive Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 | Contrastive Loss: 1.4890
Epoch 2 | Contrastive Loss: 1.3896


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：自己手写相似度矩阵时，**极其容易忘记 L2 归一化（`.norm(p=2)`）**。如果不归一化，点积的数值会失控爆炸，导致 Loss 全是 NaN。
- **⑥ 科研实验室**：对比学习对 Batch Size 有极度变态的渴求！因为 Batch Size 越大，负样本就越多，模型见识越广，能力越强。OpenAI 训练原版 CLIP 时，Batch Size 高达 32768！实验室如果要搞对比学习，通常需要动用 8 卡甚至多机多卡做分布式训练（Distributed Data Parallel, DDP）和跨卡的梯度同步（Gathering tensors across GPUs）。
- **⑦ 工程实践建议**：工业界使用时，如果有领域内的新词（比如医学专有名词），会把微调好的模型重新走一遍上面的流程（Fine-tuning），以便对齐行业词汇。

## 步骤 7-10：保存、加载与应用 (Zero-Shot 图像分类与以文搜图)

这才是 CLIP 改变世界的杀手锏！我们不需要微调任何分类头，直接把你想分类的类别硬写成句子，它就能算出相似度。这就是所谓的 **Zero-shot (零样本分类)**。

In [6]:
# 保存与加载
model.save_pretrained("./finetuned_clip")
processor.save_pretrained("./finetuned_clip")
model.eval()

# --- 惊艳的 Zero-Shot 推理实战 ---
# 假设我们拿到了一张根本没在训练集里见过的图片（这里模拟一张）
test_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

# 我们随便给它提供几个候选的文本标签 (Prompting)
candidate_labels = ["a photo of a car", "a photo of a cat", "a photo of a banana"]

# 让 Processor 一次性处理这张图片和三个候选句子
inputs = processor(text=candidate_labels, images=test_image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    # 获取 图片->文本 的相似度得分矩阵 [1, 3]
    logits_per_image = outputs.logits_per_image 
    # 使用 softmax 将得分转为总和为 100% 的概率分布
    probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

print("\n🔥 零样本 (Zero-Shot) 分类结果：")
for label, prob in zip(candidate_labels, probs):
    print(f"[{label}]: {prob * 100:.2f}%")


🔥 零样本 (Zero-Shot) 分类结果：
[a photo of a car]: 44.31%
[a photo of a cat]: 25.93%
[a photo of a banana]: 29.75%


## 步骤 11：ONNX 导出

多模态模型的导出需要分别导出双塔，也就是导出一个 `vision_model.onnx` 和一个 `text_model.onnx`。因为在工业界部署中，图片的特征通常是提前算好存起来的（离线计算），在线阶段（用户输入搜索词时），只跑文本塔的 ONNX 来实时计算查询向量。

In [7]:
# 导出文本塔
dummy_input_ids = torch.zeros(1, 16, dtype=torch.long, device=device)
dummy_attention_mask = torch.ones(1, 16, dtype=torch.long, device=device)

torch.onnx.export(
    model.text_model, 
    (dummy_input_ids, dummy_attention_mask), 
    "clip_text_tower.onnx",
    export_params=True, opset_version=14,
    input_names=['input_ids', 'attention_mask'],
    output_names=['text_features'],
    dynamic_axes={'input_ids': {0: 'batch_size', 1: 'seq_len'}, 'attention_mask': {0: 'batch_size', 1: 'seq_len'}}
)

# 导出视觉塔
dummy_pixel_values = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(
    model.vision_model, 
    dummy_pixel_values, 
    "clip_vision_tower.onnx",
    export_params=True, opset_version=14,
    input_names=['pixel_values'],
    output_names=['image_features'],
    dynamic_axes={'pixel_values': {0: 'batch_size'}}
)
print("\n✅ CLIP 双塔已成功分别导出为 ONNX，可以直接交接给部署工程团队啦！")

/var/folders/4x/gykrz2md4nz_j7l5r35_m6880000gn/T/ipykernel_34489/3542719305.py:5: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0717 16:08:04.164000 34489 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `CLIPTextTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CLIPTextTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: seq_len will not be used, since it shares the same shape constraints with another axis: seq_len.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/var/folders/4x/gykrz2md4nz_j7l5r35_m6880000gn/T/ipykernel_34489/3542719305.py:17: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0717 16:08:08.090000 34489 site-packages/torch/onnx/_internal/exporter/

[torch.onnx] Obtain model graph for `CLIPVisionTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CLIPVisionTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

✅ CLIP 双塔已成功分别导出为 ONNX，可以直接交接给部署工程团队啦！


## 🏆 第五阶段结语

对比学习（Contrastive Learning）是过去 3 年 AI 界最重要的基石思想之一，而 CLIP 则是将其发扬光大的代表作。在手搓了复杂的 InfoNCE 矩阵计算后，你已经具备了打通图像与自然语言的能力。

请运行本代码，如果 Hugging Face 下载速度较慢请耐心等待。一切跑通之后，请深呼吸——我们将进入本入门教程的最巅峰、也是当今 AI 科研最前沿的领域：**第六阶段，使用 LoRA 技术对多模态大语言模型（VLM）进行参数高效微调！**